# Inventory Transfer Optimization - Full Process Analysis

This notebook walks through the complete optimization process step by step:
1. Data loading
2. Inventory analysis and imbalance identification
3. Rule-based optimization
4. Genetic algorithm optimization
5. Comparison, impact analysis and export

## Overview
Move stock from stores with **excess** to stores that **need** it, filling as many
shortages as possible at the lowest transport cost.

## 1. Setup

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

# Make the project package `src` importable (works from the project root or notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")

## 2. Import Project Components

In [ ]:
from src import config
from src.engine.analyzer import InventoryAnalyzer
from src.engine.data_loader import find_missing_files, load_project_data
from src.engine.genetic_algorithm import GAConfig, GeneticAlgorithmOptimizer
from src.engine.results_manager import OptimizationResult, select_best
from src.engine.rule_based import RuleBasedOptimizer
from src.engine.transfer_plan import add_readable_names, summarize_plan

print(f"Min inventory days: {config.MIN_INVENTORY_DAYS}")
print(f"Max inventory days: {config.MAX_INVENTORY_DAYS}")

## 3. Data Loading and Validation

In [ ]:
missing = find_missing_files(config.DATA_DIR)
if missing:
    raise FileNotFoundError(f"Missing {missing}. Run: python src/main.py --generate-data")

data = load_project_data(config.DATA_DIR)
sales_df, inventory_df = data.sales, data.inventory
stores_df, products_df = data.stores, data.products
distance_matrix, transport_cost_matrix = data.network.distance_matrix, data.network.cost_matrix

print(f"Sales records:      {len(sales_df):,}")
print(f"Inventory records:  {len(inventory_df):,}")
print(f"Stores:             {len(stores_df):,}")
print(f"Products:           {len(products_df):,}")
print(f"Distance matrix:    {distance_matrix.shape}")
print(f"Sales date range:   {sales_df['date'].min():%Y-%m-%d} -> {sales_df['date'].max():%Y-%m-%d}")

## 4. Explore the Data

In [ ]:
for name, df in [("Sales", sales_df), ("Inventory", inventory_df), ("Stores", stores_df), ("Products", products_df)]:
    print(f"\n{name} (first 5 rows):")
    display(df.head())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(sales_df["quantity"], bins=30, alpha=0.7, color="skyblue")
axes[0, 0].set(title="Sales Quantity Distribution", xlabel="Quantity", ylabel="Frequency")

axes[0, 1].hist(inventory_df["current_stock"], bins=30, alpha=0.7, color="lightgreen")
axes[0, 1].set(title="Current Stock Distribution", xlabel="Units", ylabel="Frequency")

top_sales = sales_df.groupby("store_id")["quantity"].sum().nlargest(10)
axes[1, 0].bar(top_sales.index.astype(str), top_sales.values, color="coral")
axes[1, 0].set(title="Top 10 Stores by Units Sold", xlabel="Store ID", ylabel="Units")

top_stock = inventory_df.groupby("store_id")["current_stock"].sum().nlargest(10)
axes[1, 1].bar(top_stock.index.astype(str), top_stock.values, color="lightcoral")
axes[1, 1].set(title="Top 10 Stores by Stock", xlabel="Store ID", ylabel="Units")

plt.tight_layout()
plt.show()

## 5. Inventory Analysis

In [ ]:
analyzer = InventoryAnalyzer(sales_df, inventory_df, stores_df, products_df)
analysis_df = analyzer.analyze()
print(f"{len(analysis_df)} store-product combinations analyzed")
display(analysis_df.head())

In [ ]:
excess_df, needed_df = analyzer.identify_imbalances()

total_excess = excess_df["excess_units"].sum()
total_needed = needed_df["needed_units"].sum()
print(f"\nTotal excess units: {total_excess:,}")
print(f"Total needed units: {total_needed:,}")
print(f"Excess to needed ratio: {total_excess / total_needed:.2f}" if total_needed else "No needs")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(excess_df["excess_units"], bins=20, alpha=0.7, color="red")
axes[0, 0].set(title="Excess Units Distribution", xlabel="Excess units", ylabel="Frequency")

axes[0, 1].hist(needed_df["needed_units"], bins=20, alpha=0.7, color="orange")
axes[0, 1].set(title="Needed Units Distribution", xlabel="Needed units", ylabel="Frequency")

top_excess = excess_df.groupby("store_id")["excess_units"].sum().nlargest(10)
axes[1, 0].bar(top_excess.index.astype(str), top_excess.values, color="red", alpha=0.7)
axes[1, 0].set(title="Top 10 Stores with Excess", xlabel="Store ID", ylabel="Excess units")

top_needed = needed_df.groupby("store_id")["needed_units"].sum().nlargest(10)
axes[1, 1].bar(top_needed.index.astype(str), top_needed.values, color="orange", alpha=0.7)
axes[1, 1].set(title="Top 10 Stores Needing Stock", xlabel="Store ID", ylabel="Needed units")

plt.tight_layout()
plt.show()

### Transport cost matrix

Cost (VND) of shipping **one unit** between two stores. The diagonal (a store to
itself) is empty on purpose: it is not a valid route.

In [ ]:
display(transport_cost_matrix.iloc[:5, :5])

off_diagonal = transport_cost_matrix.where(~np.eye(len(transport_cost_matrix), dtype=bool))
nan_count = int(transport_cost_matrix.isna().sum().sum())
diagonal_nan = int(np.diag(transport_cost_matrix.isna()).sum())
print(f"Empty cells: {nan_count} (on the diagonal: {diagonal_nan})")

costs = off_diagonal.stack()
print(f"Min cost/unit:    {costs.min():>12,.0f} VND")
print(f"Median cost/unit: {costs.median():>12,.0f} VND")
print(f"Max cost/unit:    {costs.max():>12,.0f} VND")

## 6. Rule-Based Optimization

1. Sort needed items by need (biggest first).
2. For each one, take stock from stores with excess of the same product, **closest store first**.
3. Continue until the need is covered or no excess is left.

In [ ]:
rule_optimizer = RuleBasedOptimizer(data.network)
rule_plan = add_readable_names(rule_optimizer.optimize(excess_df, needed_df), stores_df, products_df)
display(rule_plan.head(10))

## 7. Genetic Algorithm Optimization

1. **Initial population** - random but valid transfer plans
2. **Fitness** - transport cost + a penalty for every needed unit left unfilled (lower is better)
3. **Selection** - tournament: the best of a few random plans becomes a parent
4. **Crossover** - swap the tails of two parent plans
5. **Mutation** - change a quantity, remove/add a transfer, or change a route
6. **Repair** - cap every plan so it never ships more than available or needed
7. **Elitism** - the best plan found so far is never lost

In [ ]:
ga_config = GAConfig()  # defaults from src/config.py
print(ga_config)

ga_optimizer = GeneticAlgorithmOptimizer(data.network, ga_config, seed=config.RANDOM_SEED)
ga_plan = add_readable_names(ga_optimizer.optimize(excess_df, needed_df), stores_df, products_df)
display(ga_plan.head(10))

In [ ]:
history = pd.DataFrame(ga_optimizer.history, columns=["generation", "best", "average"])
ax = history.plot(x="generation", y=["best", "average"], figsize=(10, 5), title="GA fitness per generation")
ax.set_ylabel("Fitness (VND, lower is better)")
plt.show()

## 8. Algorithm Comparison

In [ ]:
results = []
for optimizer, plan in [(rule_optimizer, rule_plan), (ga_optimizer, ga_plan)]:
    impact, _ = analyzer.evaluate_plan_impact(plan)
    results.append(OptimizationResult(optimizer.name, plan, impact, optimizer.execution_time))

comparison_df = pd.DataFrame(
    [
        {
            "Algorithm": r.algorithm,
            "Execution Time (s)": round(r.execution_time, 2),
            "Transfers": r.summary["transfers"],
            "Units": r.summary["units"],
            "Total Cost (VND)": r.summary["total_cost"],
            "Cost per Unit (VND)": r.summary["cost_per_unit"],
            "Needed Items Resolved": r.improvement("Reduction in Needed Items"),
        }
        for r in results
    ]
).set_index("Algorithm")
display(comparison_df.style.format("{:,.0f}", subset=["Total Cost (VND)", "Cost per Unit (VND)"]))

In [ ]:
metrics = ["Execution Time (s)", "Total Cost (VND)", "Transfers", "Cost per Unit (VND)"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, metric in zip(axes.flat, metrics):
    comparison_df[metric].plot.bar(ax=ax, color=["skyblue", "lightcoral"], rot=0, title=metric)
plt.tight_layout()
plt.show()

## 9. Best Algorithm

Same rule as `src/main.py`: the plan that resolves the most needed items wins;
ties go to the lower transport cost.

In [ ]:
best = select_best(results)
best_plan = best.plan
print(f"Best algorithm: {best.algorithm}")
print(f"Transfers: {best.summary['transfers']:,} | Units: {best.summary['units']:,} | "
      f"Cost: {best.summary['total_cost']:,.0f} VND")
display(best_plan.nlargest(10, "units")[["from_store_name", "to_store_name", "product_name", "units", "transport_cost"]])

## 10. Impact of the Best Plan

In [ ]:
display(best.impact)

## 11. Final Summary

In [ ]:
print("FINAL OPTIMIZATION SUMMARY")
print("=" * 50)
print(f"Stores analyzed:   {stores_df['store_id'].nunique()}")
print(f"Products analyzed: {products_df['product_id'].nunique()}")
print(f"Excess items:      {len(excess_df)} ({total_excess:,} units)")
print(f"Needed items:      {len(needed_df)} ({total_needed:,} units)")
for r in results:
    s = r.summary
    print(f"\n{r.algorithm}: {s['transfers']} transfers, {s['units']:,} units, "
          f"{s['total_cost']:,.0f} VND, {r.execution_time:.2f}s")
print(f"\nRecommended: {best.algorithm}")

## 12. Export Results (optional)

In [ ]:
SAVE_RESULTS = True  # set to False to skip writing files

if SAVE_RESULTS:
    import json

    config.RESULTS_DIR.mkdir(exist_ok=True)
    best_plan.to_csv(config.RESULTS_DIR / "notebook_best_transfer_plan.csv", index=False)
    comparison_df.to_csv(config.RESULTS_DIR / "notebook_algorithm_comparison.csv")
    summary = {
        "timestamp": datetime.now().isoformat(),
        "best_algorithm": best.algorithm,
        "total_cost": best.summary["total_cost"],
        "total_transfers": best.summary["transfers"],
        "total_units": best.summary["units"],
        "excess_items": len(excess_df),
        "needed_items": len(needed_df),
        "excess_units": int(total_excess),
        "needed_units": int(total_needed),
    }
    (config.RESULTS_DIR / "notebook_analysis_summary.json").write_text(json.dumps(summary, indent=2))
    print(f"Saved to {config.RESULTS_DIR}")